# 春第 4 回｜グループ集計と割合

同じ定時到着便数でも，何で割るかによって答える問いが変わります。今回は 2024 年 1 月の記録で ATL（アトランタ）か MCO（オーランド）を選び，依頼に合う割合を計算します。

共通課題は 1〜3 の約 20 分です。条件を増やして順位を逆転させることが目的ではありません。**何を分母に選び，その率が何を表すか**を説明してください。末尾の追加課題は任意です。

## 準備

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

最初のセルはファイルの置き場所に対応する準備です。そのまま実行してください。出力は `32347` です。

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

file_name = "nyc-flights-2024-01.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

flights = pd.read_csv(csv_path)
len(flights)

次の判定は本文と同じです。欠航せず，到着先も変更されず，到着の遅れが 15 分未満なら `True` にします。この判定を足したものが，今回の二つの依頼に共通する分子です。

In [ ]:
flights["on_time"] = (
    (flights["cancelled"] == 0)
    & (flights["diverted"] == 0)
    & (flights["arrival_delay"] < 15)
)
flights[["origin", "arrival_delay", "on_time"]].head()

## 1. 依頼と行先を選ぶ（5 分）

行先は ATL か MCO，依頼は A か B を一つずつ選びます。

| 依頼 | 知りたいこと |
|---|---|
| A | この行先へ出る予定だった便のうち，何割が定時に目的地へ着いたか。 |
| B | この行先への到着の遅れが記録された便に限ると，何割が定時だったか。 |

選んだ依頼（A／B）：

分母に含める便（自分の言葉で）：

次の `destination` に行先を指定して，集計の材料を表示します。`scheduled` は予定便数，`recorded` は到着の遅れが記録された便数，`on_time` は定時到着便数です。空港を一つ選び，この 3 列の値を読み比べてください。

In [ ]:
destination = "ATL"  # MCO を選んだ人はここを変更
route = flights.loc[flights["destination"] == destination]
groups = route.groupby("origin")

counts = groups.size().to_frame("scheduled")
counts["recorded"] = groups["arrival_delay"].count()
counts["on_time"] = groups["on_time"].sum()
counts

## 2. 分母を選んで計算する（9 分）

`groups` は，選んだ行先の便を出発空港ごとにまとめたものです。依頼に合う分母を，`denominator = ...` という一行で書きます。本文の `size()` と `count()` を参照できます。

書いたら，その下の 4 行も先頭の `#` を外して実行してください。`denominator` と `on_time` は，同じ空港名の行どうしで割り算されます。元の `counts` は残したまま，結果を `report` にまとめます。

`count()` を使う場合は，**どの列の値の有無を数えるか**にも注意してください。`on_time` は，元の到着の遅れが欠損でも `False` が入る列です。

In [ ]:
# ここに denominator = ... の一行を書く

# report = denominator.to_frame("denominator")
# report["on_time"] = counts["on_time"]
# report["rate"] = report["on_time"] / report["denominator"]
# report.round({"rate": 4})

<details>
<summary>分母の式のヒント</summary>

全予定便を数えるなら `groups.size()`，到着の遅れに値がある便を数えるなら `groups["arrival_delay"].count()` です。自分が選んだ依頼に合う方を `denominator` へ代入します。

</details>

結果の表ができたら，次の行の `#` を外して図を描きます。縦軸の `0.8` は 80% です。図だけでは分母の定義や件数は伝わらないため，上の表と合わせて読みます。

In [ ]:
# report["rate"].plot.bar(ylim=(0, 1))
# plt.show()

## 3. 数字を説明する（6 分）

一つの空港を選び，分子と分母を使って結果を一文にしてください。別の人がこの文だけを読んでも，期間・行先・分母が分かるように書きます。

- 見出し：
- 根拠：定時到着 ____ 便／分母 ____ 便，約 ____ %。
- その分母を選んだ理由：
- もう一方の依頼なら，分母に含める便はどう変わるか：

例えば「〇〇空港がよい」だけでは，何を比べたかが残りません。「2024 年 1 月の ____ 行きでは，____ 便のうち約 ____ % が定時に到着した」のように，数えた対象を入れます。

この結果を別の月の旅行に使う場合，さらに何を調べたいですか。今月の割合を計算したことと，今後の便の成績を予測したことは区別します。

追加で調べたいこと：

**提出前に**，ノートを最初から実行して保存します。選んだ行先・依頼，記入した分母の式，実行結果と上の説明を含めたノートを提出してください。任意の追加課題は，コメントのままで構いません。

## 追加課題（任意）

もう一方の依頼の率も計算し，同じ空港について比べます。`report` を上書きせず，別の名前へ保存してください。変わったのは定時到着便数か，分母か，両方かを説明します。順位が変わらなくても，差や件数から説明できます。

本文の重みの比較へ進みたい人は，MCO と SFO の率を半分ずつ重く見た場合と，実際の予定便数でまとめた場合を比べ，それぞれが何を答えるかを言葉にしてください。こちらは共通課題の提出には不要です。

In [ ]:
# 追加で調べるコードをここに書く（任意）